# RK1828 图像分类 Demo（MobileNetV2 / ResNet50-v2）

本 notebook 在 **RK1828 AI 协处理器**（PCIe 连接，20 TOPS NPU）上运行 ImageNet 图像分类，
完整演示 AI 推理的标准流程：

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 选择模型和图片 |
| 3 | 前处理（把图片变成模型能吃的张量） |
| 4 | RKNN 推理（在 NPU 上运行） |
| 5 | 后处理（反量化 + Softmax，输出 Top-5） |
| 6 | 多模型对比（2 模型 × 2 量化全量跑一遍） |

> 小知识：**i8** 是把权重压成 8bit 整数的量化版本，更小更快；
> **fp16** 是 16bit 浮点版本。两种都可以跑一遍对比结果。

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。


本 demo 会下载 MobileNetV2 和 ResNet50-v2 两个模型（各 i8/fp16 两版），以及 ImageNet 标签和示例图片。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("mobilenet_v2-rknn3-rk1828", "i8/mobilenet_v2.rknn", "models/cnn/mobilenet_v2/i8/mobilenet_v2.rknn"),
    ("mobilenet_v2-rknn3-rk1828", "i8/mobilenet_v2.weight", "models/cnn/mobilenet_v2/i8/mobilenet_v2.weight"),
    ("mobilenet_v2-rknn3-rk1828", "fp16/mobilenet_v2.rknn", "models/cnn/mobilenet_v2/fp16/mobilenet_v2.rknn"),
    ("mobilenet_v2-rknn3-rk1828", "fp16/mobilenet_v2.weight", "models/cnn/mobilenet_v2/fp16/mobilenet_v2.weight"),
    ("mobilenet_v2-rknn3-rk1828", "imagenet_labels.txt", "datasets/imagenet_labels.txt"),
    ("mobilenet_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
    ("resnet50_v2-rknn3-rk1828", "i8/resnet50_v2.rknn", "models/cnn/resnet50_v2/i8/resnet50_v2.rknn"),
    ("resnet50_v2-rknn3-rk1828", "i8/resnet50_v2.weight", "models/cnn/resnet50_v2/i8/resnet50_v2.weight"),
    ("resnet50_v2-rknn3-rk1828", "fp16/resnet50_v2.rknn", "models/cnn/resnet50_v2/fp16/resnet50_v2.rknn"),
    ("resnet50_v2-rknn3-rk1828", "fp16/resnet50_v2.weight", "models/cnn/resnet50_v2/fp16/resnet50_v2.weight"),
    ("resnet50_v2-rknn3-rk1828", "imagenet_labels.txt", "datasets/imagenet_labels.txt"),
    ("resnet50_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 选择模型和图片

- **模型**：`mobilenet_v2`（轻量，3.4M 参数）或 `resnet50_v2`（经典，25M 参数）
- **量化版本**：`i8`（int8 量化）或 `fp16`（半精度浮点）
- **图片**：`datasets/` 里的示例图（也可以自己放 224×224 左右的图片进去）

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(options=["mobilenet_v2", "resnet50_v2"],
                           description="模型:")
w_quant = widgets.Dropdown(options=["i8", "fp16"], description="量化:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="图片:", layout=widgets.Layout(width="60%"))
display(w_model, w_quant, w_image)

LABELS = open("datasets/imagenet_labels.txt").read().splitlines()
print(f"{len(LABELS)} 个类别。选好后继续往下运行 ▼")

## 3. 前处理

模型输入是 **224×224 NHWC uint8 RGB**（归一化已烘焙进模型，无需减均值除方差）。
我们把图片缩放到 224×224 并加上 batch 维度。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import setup_cjk_font

setup_cjk_font()
img_orig = Image.open(w_image.value).convert("RGB")
img = img_orig.resize((224, 224))
x = np.asarray(img)[None]  # [1,224,224,3] uint8 NHWC
print("输入张量:", x.shape, x.dtype)

# 前处理每个阶段都画出来：原图 -> resize -> 输入张量
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, im, title in [
    (axes[0], img_orig, f"1. 原图 {img_orig.size[0]}x{img_orig.size[1]}"),
    (axes[1], img, "2. resize 224x224"),
    (axes[2], x[0], f"3. 模型输入 {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN 推理

加载 `.rknn`（计算图）+ `.weight`（权重）到 RK1828，执行一次推理并计时。

In [ ]:
import time
from rknn3 import Rknn3Model

model_path = f"models/cnn/{w_model.value}/{w_quant.value}/{w_model.value}.rknn"
model = Rknn3Model(model_path)
print("输入属性:", model.input_attrs)
print("输出属性:", model.output_attrs)

_ = model.infer([x])  # 预热
t0 = time.time()
outputs = model.infer([x])
print(f"推理耗时: {(time.time()-t0)*1000:.1f} ms")

## 5. 后处理：反量化 + Softmax

i8 模型的输出是 int8，要用输出张量自带的 `scale` / `zero_point` 还原成浮点 logits；
再 Softmax 取 Top-5。

In [ ]:
from vision_post import dequant

attr = model.output_attrs[0]
logits = dequant(outputs[0], attr).ravel()
probs = np.exp(logits - logits.max())
probs /= probs.sum()

top5 = probs.argsort()[::-1][:5]
for i in top5:
    print(f"{probs[i]*100:5.1f}%  {LABELS[i]}")

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1.6]})
axes[0].imshow(img)
axes[0].set_title(f"{w_model.value} ({w_quant.value})")
axes[0].axis("off")
names = [LABELS[i] for i in top5][::-1]
axes[1].barh(names, [probs[i] for i in top5][::-1], color="#3b82f6")
axes[1].set_title("Top-5 概率")
axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.show()

model.close()

## 6. 多模型对比（i8 + fp16 全量）

同一张图，把 **2 个模型 × 2 种量化（i8 / fp16）共 4 组**全部跑一遍。
运行过程实时打印进度；全部完成后输出对比表格、耗时柱状图和 Top-5 效果拼图。
每个模型计时 3 次取最快（排除首次预热抖动）。

In [ ]:
import time
from rknn3 import Rknn3Model
from vision_post import dequant

ALL_MODELS = ["mobilenet_v2", "resnet50_v2"]
QUANTS = ["i8", "fp16"]

compare = []  # (model, quant, latency_ms, top5_idx, top5_prob)
total = len(ALL_MODELS) * len(QUANTS)
step = 0
for name in ALL_MODELS:
    for quant in QUANTS:
        step += 1
        print(f"[{step}/{total}] 加载 {name} ({quant}) ...", flush=True)
        m = Rknn3Model(f"models/cnn/{name}/{quant}/{name}.rknn")
        m.infer([x])  # 预热
        best, outs = float("inf"), None
        for _ in range(3):
            t0 = time.time()
            outs = m.infer([x])
            best = min(best, (time.time() - t0) * 1000)
        logits = dequant(outs[0], m.output_attrs[0]).ravel()
        p = np.exp(logits - logits.max())
        p /= p.sum()
        top5_i = p.argsort()[::-1][:5]
        compare.append((name, quant, best, top5_i, p[top5_i]))
        m.close()
        print(f"      -> {best:6.1f} ms, Top-1: {LABELS[top5_i[0]]} "
              f"({p[top5_i[0]]*100:.1f}%)", flush=True)
print("\n全部完成 ✅", flush=True)

### 对比表格

In [ ]:
from IPython.display import HTML

rows = "".join(
    f"<tr><td>{n}</td><td>{q}</td><td>{lat:.1f}</td>"
    f"<td>{', '.join(f'{LABELS[i]}({pp*100:.1f}%)' for i, pp in list(zip(t, pr))[:5])}</td></tr>"
    for n, q, lat, t, pr in compare)
HTML("<table style='text-align:left'>"
     "<tr><th>模型</th><th>量化</th><th>耗时(ms)</th><th>Top-5（概率）</th></tr>"
     + rows + "</table>")

### 耗时柱状图（i8 vs fp16）

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
colors = {"i8": "#3b82f6", "fp16": "#f59e0b"}
xs = np.arange(len(ALL_MODELS))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, q in enumerate(QUANTS):
    lats = [lat for n, qq, lat, *_ in compare if qq == q]
    tops = [LABELS[t[0]] for n, qq, _, t, _ in compare if qq == q]
    bars = ax.bar(xs + (k - 0.5) * width, lats, width, label=q, color=colors[q])
    for bar, lat, top in zip(bars, lats, tops):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{lat:.0f}ms\n{top}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs, ALL_MODELS)
ax.set_ylabel("推理耗时 (ms)")
ax.set_title("分类模型对比（同一张图，i8 vs fp16）")
ax.set_ylim(0, max(lat for _, _, lat, *_ in compare) * 1.35)
ax.legend()
plt.tight_layout()
plt.show()

### Top-5 效果拼图

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (n, q, lat, t, pr) in zip(axes.flat, compare):
    names = [LABELS[i] for i in t][::-1]
    ax.barh(names, list(pr)[::-1], color="#3b82f6" if q == "i8" else "#f59e0b")
    ax.set_xlim(0, 1)
    ax.set_title(f"{n} ({q})  {lat:.0f}ms", fontsize=11)
plt.tight_layout()
plt.show()

## 小结

- 完整流程：**下载模型 → 前处理 → NPU 推理 → 反量化/Softmax → Top-5**
- 从第 6 步的对比可以看到：
  - **mobilenet_v2**（轻量，3.4M 参数）明显比 **resnet50_v2**（经典，25M 参数）快；
    ResNet50-v2 精度更高（ImageNet Top-1 约 72% vs 75.6%，论文值）。
  - 同一模型的 i8 通常比 fp16 更快，Top-5 结果基本一致——量化带来的精度损失很小。